In [1]:
# Requirements
!pip install pandas numpy scikit-learn tensorflow torch transformers datasets textstat xgboost
!pip install imbalanced-learn
!pip install nlpaug
!pip install sentencepiece
!pip install tf-keras
!pip install evaluate 

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/igraph-0.11.8-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.11.8-py3.12.egg is deprecated. pip 25

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, cohen_kappa_score, f1_score
import torch
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, AutoModelForSequenceClassification, EarlyStoppingCallback
from transformers import TFBertPreTrainedModel, TFBertMainLayer, InputFeatures
from evaluate import load  

2025-04-03 16:33:16.900598: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743697996.919922    3359 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743697996.925968    3359 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743697996.942099    3359 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743697996.942114    3359 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1743697996.942116    3359 computation_placer.cc:177] computation placer alr

In [3]:
data = pd.read_csv("sample_full.csv")

In [4]:
data.fillna({'Remember': 0, 'Understand': 0, 'Apply': 0, 'Analyze': 0, 'Evaluate': 0, 'Create':0}, inplace=True)

In [5]:
LIWC_data = pd.read_csv("Learning_outcomes.csv")
data = data.join(LIWC_data).drop(['A'], axis=1)

In [6]:
data.head()

,Learning_outcome,Remember,Understand,Apply,Analyze,Evaluate,Create,WC,Analytic,Clout,...,Comma,Colon,SemiC,QMark,Exclam,Dash,Quote,Apostro,Parenth,OtherP
0,Analyze the health economic implications of e...,0.0,0.0,0.0,1.0,0.0,0.0,9,99.00,50.00,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0
1,Apply research skills to operate effectively ...,0.0,0.0,1.0,0.0,0.0,0.0,14,99.00,92.33,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0
2,Assess and synthesise diverse information abo...,0.0,0.0,0.0,0.0,1.0,1.0,26,43.96,77.92,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0
3,Describe the general characteristics of the m...,0.0,1.0,0.0,0.0,0.0,0.0,23,99.00,50.00,...,8.7,0.0,0.0,0.0,0.0,4.35,0.0,0.0,0.0,0.0
4,Evaluate the different models of perioperativ...,0.0,0.0,0.0,0.0,1.0,0.0,10,98.58,15.86,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0


In [7]:
labels = data[data.columns[1:7]].values.tolist()

In [8]:
data.columns[1:7]

Index(['Remember', 'Understand', 'Apply', 'Analyze', 'Evaluate', 'Create'], dtype='object')

In [9]:
data

,Learning_outcome,Remember,Understand,Apply,Analyze,Evaluate,Create,WC,Analytic,Clout,...,Comma,Colon,SemiC,QMark,Exclam,Dash,Quote,Apostro,Parenth,OtherP
0,Analyze the health economic implications of e...,0.0,0.0,0.0,1.0,0.0,0.0,9,99.00,50.00,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
1,Apply research skills to operate effectively ...,0.0,0.0,1.0,0.0,0.0,0.0,14,99.00,92.33,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
2,Assess and synthesise diverse information abo...,0.0,0.0,0.0,0.0,1.0,1.0,26,43.96,77.92,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
3,Describe the general characteristics of the m...,0.0,1.0,0.0,0.0,0.0,0.0,23,99.00,50.00,...,8.70,0.0,0.0,0.0,0.0,4.35,0.0,0.0,0.0,0.00
4,Evaluate the different models of perioperativ...,0.0,0.0,0.0,0.0,1.0,0.0,10,98.58,15.86,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21375,"Write/type simple sentences using hiragana, ka...",0.0,0.0,1.0,0.0,0.0,0.0,11,80.14,50.00,...,18.18,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,9.09
21376,Writing of assessment reports and giving feedb...,0.0,0.0,0.0,0.0,1.0,0.0,9,98.87,86.68,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
21377,You will develop the ability to work in a team...,0.0,0.0,1.0,0.0,0.0,0.0,15,99.00,97.69,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00
21378,You will develop their oral presentation skill...,0.0,0.0,0.0,0.0,0.0,1.0,33,93.26,96.52,...,6.06,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00


# Machine Learning Models

In [10]:
import textstat

In [11]:
def generateX(data_x, test_x, textual_column_index, start_index_LIWC, end_index_LIWC):
    column_names = []
    print("Getting Unigram...")
    uni_cv = CountVectorizer(stop_words='english', ngram_range=(1, 1), max_features=1000)
    unigram = uni_cv.fit_transform(data_x[:, textual_column_index])
    unigram = unigram.toarray()
    unigram_test = uni_cv.transform(test_x[:,textual_column_index]).toarray()
    temp = uni_cv.get_feature_names_out().tolist()
    column_names += ["uni_"+name for name in temp]
    print("Getting Bigram...")
    bi_cv = CountVectorizer(stop_words='english', ngram_range=(2, 2), max_features=1000)
    bigram = bi_cv.fit_transform(data_x[:, textual_column_index])
    bigram = bigram.toarray()
    bigram_test = bi_cv.transform(test_x[:, textual_column_index]).toarray()
    temp = bi_cv.get_feature_names_out().tolist()
    column_names += ["bi_"+name for name in temp]
    print("Getting Tfidf...")
    tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 1), max_features=1000)
    t = tfidf.fit_transform(data_x[:, textual_column_index])
    t = t.toarray()
    t_test = tfidf.transform(test_x[:, textual_column_index]).toarray()
    temp = tfidf.get_feature_names_out().tolist()
    column_names += ["tfidf_"+name for name in temp]
    print("Getting ARI...")
    ari = [textstat.automated_readability_index(text) for text in data_x[:, textual_column_index]]
    ari_test = [textstat.automated_readability_index(text) for text in test_x[:, textual_column_index]]
    column_names.append("ari")
    combined_data_x = []
    combined_test_x = []
    print("Combining...")
    for i in range(len(data_x)):
        combined_data_x.append(unigram[i].tolist()
                              + bigram[i].tolist()
                              + t[i].tolist()
                              + [ari[i]]
                              + data_x[i, start_index_LIWC:end_index_LIWC].tolist())
    for i in range(len(test_x)):
        combined_test_x.append(unigram_test[i].tolist()
                              + bigram_test[i].tolist()
                              + t_test[i].tolist()
                              + [ari_test[i]]
                              + test_x[i, start_index_LIWC:end_index_LIWC].tolist())
    print("Generated feature shape is", np.array(combined_data_x).shape)
    print("Generated test feature is", np.array(combined_test_x).shape)
    return combined_data_x, column_names, combined_test_x

In [12]:
data.drop(columns=list(data.columns[1:7])).iloc[:, 0]

0         Analyze the health economic implications of e...
1         Apply research skills to operate effectively ...
2         Assess and synthesise diverse information abo...
3         Describe the general characteristics of the m...
4         Evaluate the different models of perioperativ...
                               ...                        
21375    Write/type simple sentences using hiragana, ka...
21376    Writing of assessment reports and giving feedb...
21377    You will develop the ability to work in a team...
21378    You will develop their oral presentation skill...
21379    You will gain an ability to use geoscientific ...
Name: Learning_outcome, Length: 21380, dtype: object

In [13]:
train_x, test_x, train_y, test_y = train_test_split(data.drop(columns=list(data.columns[1:8])), data[data.columns[1:7]], test_size=0.2, random_state=42)

In [14]:
np.unique(train_y['Remember'].tolist(), return_counts=True), np.unique(test_y['Remember'].tolist(), return_counts=True)

((array([0., 1.]), array([16142,   962])),
 (array([0., 1.]), array([4053,  223])))

In [15]:
np.unique(train_y['Understand'].tolist(), return_counts=True), np.unique(test_y['Understand'].tolist(), return_counts=True)

((array([0., 1.]), array([12448,  4656])),
 (array([0., 1.]), array([3107, 1169])))

In [16]:
np.unique(train_y['Apply'].tolist(), return_counts=True), np.unique(test_y['Apply'].tolist(), return_counts=True)

((array([0., 1.]), array([12229,  4875])),
 (array([0., 1.]), array([3070, 1206])))

In [17]:
np.unique(train_y['Analyze'].tolist(), return_counts=True), np.unique(test_y['Analyze'].tolist(), return_counts=True)

((array([0., 1.]), array([14306,  2798])),
 (array([0., 1.]), array([3615,  661])))

In [18]:
np.unique(train_y['Evaluate'].tolist(), return_counts=True), np.unique(test_y['Evaluate'].tolist(), return_counts=True)

((array([0., 1.]), array([14076,  3028])),
 (array([0., 1.]), array([3470,  806])))

In [19]:
np.unique(train_y['Create'].tolist(), return_counts=True), np.unique(test_y['Create'].tolist(), return_counts=True)

((array([0., 1.]), array([14005,  3099])),
 (array([0., 1.]), array([3488,  788])))

In [20]:
one_hot = []
for d in data[data.columns[1:7]].values:
    one_hot.append(np.array2string(d).count("1"))
np.unique(one_hot, return_counts=True)

(array([1, 2, 3, 4]), array([18773,  2325,   280,     2]))

In [21]:
ml_train_x, column_names, ml_test_x = generateX(train_x.to_numpy(), test_x.to_numpy(), 0, 1, 94)

Getting Unigram...
Getting Bigram...
Getting Tfidf...
Getting ARI...
Combining...
Generated feature shape is (17104, 3093)
Generated test feature is (4276, 3093)


In [22]:
column_names += data.columns[7:].tolist()

In [23]:
rf = RandomForestClassifier()
rf.fit(ml_train_x, train_y)

RandomForestClassifier()

In [24]:
pred_y = rf.predict(ml_test_x)

In [25]:
print(classification_report(test_y, pred_y, output_dict=False, target_names=list(data.columns[1:7]), digits=3))

              precision    recall  f1-score   support

    Remember      0.869     0.682     0.764       223
  Understand      0.945     0.768     0.848      1169
       Apply      0.942     0.780     0.854      1206
     Analyze      0.964     0.735     0.834       661
    Evaluate      0.955     0.738     0.833       806
      Create      0.930     0.689     0.792       788

   micro avg      0.943     0.745     0.832      4853
   macro avg      0.934     0.732     0.821      4853
weighted avg      0.943     0.745     0.832      4853
 samples avg      0.777     0.755     0.762      4853



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
pred_score_y = rf.predict_proba(ml_test_x)

In [27]:
np.array(test_x).shape

(4276, 93)

In [28]:
pred_score_y = np.transpose([score[:, 1] for score in rf.predict_proba(ml_test_x)])

In [29]:
roc_auc_score(test_y, pred_score_y, average=None)

array([0.9837993 , 0.96962597, 0.97016046, 0.98036882, 0.97711615,
       0.97157513])

In [30]:
f1_score(test_y, pred_y, average="micro")

0.8321823204419889

In [31]:
accuracy_score(test_y, pred_y)

0.7282507015902713

In [32]:
ml_result_df = pd.DataFrame(data=pred_y, columns=data.columns[1:7])

In [33]:
ml_result_df

,Remember,Understand,Apply,Analyze,Evaluate,Create
0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
4271,0.0,0.0,1.0,0.0,1.0,0.0
4272,0.0,1.0,0.0,0.0,0.0,0.0
4273,0.0,0.0,0.0,0.0,0.0,1.0
4274,0.0,0.0,1.0,0.0,0.0,0.0


In [34]:
ml_golden_df = pd.DataFrame(data=test_y, columns=data.columns[1:7])

In [35]:
print(accuracy_score(ml_golden_df['Remember'].tolist(), ml_result_df['Remember'].tolist()))
print(accuracy_score(ml_golden_df['Understand'].tolist(), ml_result_df['Understand'].tolist()))
print(accuracy_score(ml_golden_df['Apply'].tolist(), ml_result_df['Apply'].tolist()))
print(accuracy_score(ml_golden_df['Analyze'].tolist(), ml_result_df['Analyze'].tolist()))
print(accuracy_score(ml_golden_df['Evaluate'].tolist(), ml_result_df['Evaluate'].tolist()))
print(accuracy_score(ml_golden_df['Create'].tolist(), ml_result_df['Create'].tolist()))

0.9780168381665107
0.9244621141253508
0.9244621141253508
0.9548643592142189
0.9441066417212348
0.9331150608044901


In [36]:
print(cohen_kappa_score(ml_golden_df['Remember'].tolist(), ml_result_df['Remember'].tolist()))
print(cohen_kappa_score(ml_golden_df['Understand'].tolist(), ml_result_df['Understand'].tolist()))
print(cohen_kappa_score(ml_golden_df['Apply'].tolist(), ml_result_df['Apply'].tolist()))
print(cohen_kappa_score(ml_golden_df['Analyze'].tolist(), ml_result_df['Analyze'].tolist()))
print(cohen_kappa_score(ml_golden_df['Evaluate'].tolist(), ml_result_df['Evaluate'].tolist()))
print(cohen_kappa_score(ml_golden_df['Create'].tolist(), ml_result_df['Create'].tolist()))

0.7524667477112301
0.798070220171983
0.8032270160729632
0.8087555489227513
0.7998553894457796
0.7527568148073869


# Bert

In [37]:
class EncodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [38]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=6)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [39]:
train_x, test_x, train_y, test_y = train_test_split(data['Learning_outcome'].tolist(), labels, test_size=0.2, random_state=42)
train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

In [40]:
train_encoded = tokenizer(train_x, truncation=True, padding=True, max_length=100)
val_encoded = tokenizer(val_x, truncation=True, padding=True, max_length=100)
test_encoded = tokenizer(test_x, truncation=True, padding=True, max_length=100)

In [41]:
train_set, val_set, test_set = EncodeDataset(train_encoded, train_y), EncodeDataset(val_encoded, val_y), EncodeDataset(test_encoded, test_y)

In [42]:
training_args = TrainingArguments(
        output_dir='Classification',          # output directory
        overwrite_output_dir=True,
        num_train_epochs=3,              # total number of training epochs
        per_device_train_batch_size=64,  # batch size per device during training
        per_device_eval_batch_size=64,   # batch size for evaluation
        warmup_steps=5,                # number of warmup steps for learning rate scheduler
        weight_decay=0.05,               # strength of weight decay
        logging_dir='./logs',            # directory for storing logs
        logging_steps=10,
        evaluation_strategy="steps",
        save_strategy="steps",
        save_steps=10,
        load_best_model_at_end=True
    )

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [43]:
def getClassResult(predicted):
    """Convert sigmoid outputs to binary predictions (0 or 1) with threshold 0.5"""
    return (predicted.numpy() > 0.5).astype(int)  

f1_metric = load("f1")  
accuracy_metric = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = tf.keras.activations.sigmoid(logits)  
    predicted = getClassResult(predictions)
    

    metrics = {
        "micro_f1": f1_metric.compute(
            predictions=predicted, 
            references=labels, 
            average="micro"
        )["f1"],
        "macro_f1": f1_metric.compute(
            predictions=predicted,
            references=labels,
            average="macro"
        )["f1"],
        "accuracy": accuracy_metric.compute(
            predictions=predicted,
            references=labels
        )["accuracy"]
    }
    
    class_names = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
    for i, name in enumerate(class_names):
        metrics[f"{name}_f1"] = f1_metric.compute(
            predictions=predicted[:, i],
            references=labels[:, i],
            average="binary"
        )["f1"]
    
    return metrics

In [44]:
trainer = Trainer(model=model, args=training_args, train_dataset=train_set, eval_dataset=val_set, callbacks=[EarlyStoppingCallback(early_stopping_patience=5)])

In [45]:
trainer.train()

Step,Training Loss,Validation Loss
10,0.643200,0.517182
20,0.478000,0.438626
30,0.424500,0.387627
40,0.373600,0.332283
50,0.314900,0.287630
60,0.283500,0.260861
70,0.250100,0.224306
80,0.216200,0.201726
90,0.208600,0.181473
100,0.172900,0.173207


TrainOutput(global_step=590, training_loss=0.14162694912845805, metrics={'train_runtime': 261.6512, 'train_samples_per_second': 156.884, 'train_steps_per_second': 2.454, 'total_flos': 1939177564322400.0, 'train_loss': 0.14162694912845805, 'epoch': 2.7570093457943923})

In [46]:
logits = trainer.predict(test_set)

In [47]:
logits.predictions.shape

(4276, 6)

In [48]:
predicted = tf.keras.activations.sigmoid(logits.predictions)

I0000 00:00:1743698309.533554    3359 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 73418 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:87:00.0, compute capability: 8.0


In [49]:
predicted.numpy()

array([[0.01289487, 0.01747993, 0.01571936, 0.01553479, 0.0112361 ,
        0.9781286 ],
       [0.00702958, 0.00893533, 0.00884059, 0.00917042, 0.9422553 ,
        0.14521691],
       [0.0070438 , 0.01497447, 0.9815716 , 0.01018003, 0.00727463,
        0.0131345 ],
       ...,
       [0.01152668, 0.01522569, 0.01931414, 0.01266   , 0.01079908,
        0.9777303 ],
       [0.00827708, 0.00978051, 0.9866881 , 0.01706837, 0.00894078,
        0.01790421],
       [0.01669986, 0.01058174, 0.01958135, 0.94437116, 0.9642086 ,
        0.03184412]], dtype=float32)

In [50]:
predicted_label = getClassResult(predicted)

In [52]:
count = 0
for pred in predicted_label:
    if np.count_nonzero(pred == 1) > 1:
        count += 1

In [53]:
print(classification_report(test_y, predicted_label, output_dict=False, target_names=list(data.columns[1:7]), digits=3))

              precision    recall  f1-score   support

    Remember      0.883     0.879     0.881       223
  Understand      0.954     0.917     0.935      1169
       Apply      0.938     0.925     0.931      1206
     Analyze      0.913     0.926     0.920       661
    Evaluate      0.944     0.921     0.932       806
      Create      0.908     0.904     0.906       788

   micro avg      0.932     0.917     0.924      4853
   macro avg      0.923     0.912     0.917      4853
weighted avg      0.932     0.917     0.924      4853
 samples avg      0.932     0.928     0.925      4853



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [54]:
roc_auc_score(test_y, predicted.numpy(), average=None)

array([0.99620278, 0.98491527, 0.9852934 , 0.98834366, 0.98722531,
       0.98303502])

In [55]:
accuracy_score(np.array(test_y), predicted_label)

0.8809635173058934

In [56]:
dl_result_df = pd.DataFrame(data=predicted_label, columns=data.columns[1:7])

In [57]:
print(accuracy_score(ml_golden_df['Remember'].tolist(), dl_result_df['Remember'].tolist()))
print(accuracy_score(ml_golden_df['Understand'].tolist(), dl_result_df['Understand'].tolist()))
print(accuracy_score(ml_golden_df['Apply'].tolist(), dl_result_df['Apply'].tolist()))
print(accuracy_score(ml_golden_df['Analyze'].tolist(), dl_result_df['Analyze'].tolist()))
print(accuracy_score(ml_golden_df['Evaluate'].tolist(), dl_result_df['Evaluate'].tolist()))
print(accuracy_score(ml_golden_df['Create'].tolist(), dl_result_df['Create'].tolist()))

0.9876052385406923
0.965154349859682
0.9614125350795135
0.974976613657624
0.9747427502338635
0.9653882132834425


In [59]:
from transformers import pipeline

# Create a prediction pipeline (assuming you're using a text classification model)
classifier = pipeline("text-classification", 
                     model=model, 
                     tokenizer=tokenizer, 
                     device=0 if torch.cuda.is_available() else -1)

def predict_sentence(sentence):
    # Get model outputs
    outputs = classifier(sentence, truncation=True, padding=True, return_all_scores=True)
    
    # Extract probabilities and apply threshold
    predicted_probs = [score['score'] for score in outputs[0]]
    predicted_classes = (np.array(predicted_probs) > 0.5).astype(int)
    
    # Map to class names
    class_names = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
    results = []
    for i, class_name in enumerate(class_names):
        if predicted_classes[i] == 1:
            results.append((class_name, predicted_probs[i]))
    
    return {
        "sentence": sentence,
        "predicted_classes": results,
        "all_probabilities": {name: prob for name, prob in zip(class_names, predicted_probs)}
    }

# Example usage
sentence = "Explain the main concepts of machine learning"
prediction = predict_sentence(sentence)
print(prediction)

Device set to use cuda:0


{'sentence': 'Explain the main concepts of machine learning', 'predicted_classes': [('Understand', 0.9844189286231995)], 'all_probabilities': {'Remember': 0.009317835792899132, 'Understand': 0.9844189286231995, 'Apply': 0.01362962182611227, 'Analyze': 0.008345779962837696, 'Evaluate': 0.012262539006769657, 'Create': 0.010055847465991974}}


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [61]:
from transformers import pipeline

# Create a prediction pipeline (assuming you're using a text classification model)
classifier = pipeline("text-classification", 
                     model=model, 
                     tokenizer=tokenizer, 
                     device=0 if torch.cuda.is_available() else -1)

def predict_sentence(sentence):
    # Get model outputs
    outputs = classifier(sentence, truncation=True, padding=True, return_all_scores=True)
    
    # Extract probabilities and apply threshold
    predicted_probs = [score['score'] for score in outputs[0]]
    predicted_classes = (np.array(predicted_probs) > 0.5).astype(int)
    
    # Map to class names
    class_names = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
    results = []
    for i, class_name in enumerate(class_names):
        if predicted_classes[i] == 1:
            results.append((class_name, predicted_probs[i]))
    
    return {
        "sentence": sentence,
        "predicted_classes": results,
        "all_probabilities": {name: prob for name, prob in zip(class_names, predicted_probs)}
    }

# Example usage
sentence = "Implement RL algorithms, including Dynamic Programming, Monte Carlo methods, Temporal Difference learning, Q-learning, and Deep Q Networks (DQN)"
prediction = predict_sentence(sentence)
print(prediction)

Device set to use cuda:0


{'sentence': 'Implement RL algorithms, including Dynamic Programming, Monte Carlo methods, Temporal Difference learning, Q-learning, and Deep Q Networks (DQN)', 'predicted_classes': [('Apply', 0.9873563051223755)], 'all_probabilities': {'Remember': 0.008340348489582539, 'Understand': 0.0116721261292696, 'Apply': 0.9873563051223755, 'Analyze': 0.013196311891078949, 'Evaluate': 0.009095161221921444, 'Create': 0.014529487118124962}}


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [62]:
from transformers import pipeline

# Create a prediction pipeline (assuming you're using a text classification model)
classifier = pipeline("text-classification", 
                     model=model, 
                     tokenizer=tokenizer, 
                     device=0 if torch.cuda.is_available() else -1)

def predict_sentence(sentence):
    # Get model outputs
    outputs = classifier(sentence, truncation=True, padding=True, return_all_scores=True)
    
    # Extract probabilities and apply threshold
    predicted_probs = [score['score'] for score in outputs[0]]
    predicted_classes = (np.array(predicted_probs) > 0.5).astype(int)
    
    # Map to class names
    class_names = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
    results = []
    for i, class_name in enumerate(class_names):
        if predicted_classes[i] == 1:
            results.append((class_name, predicted_probs[i]))
    
    return {
        "sentence": sentence,
        "predicted_classes": results,
        "all_probabilities": {name: prob for name, prob in zip(class_names, predicted_probs)}
    }

# Example usage
sentence = "Analyze trade-offs between model-based and model-free RL, and policy-based vs. value-based approaches."
prediction = predict_sentence(sentence)
print(prediction)

Device set to use cuda:0


{'sentence': 'Analyze trade-offs between model-based and model-free RL, and policy-based vs. value-based approaches.', 'predicted_classes': [('Analyze', 0.979082465171814)], 'all_probabilities': {'Remember': 0.01272929273545742, 'Understand': 0.01741081289947033, 'Apply': 0.012700620107352734, 'Analyze': 0.979082465171814, 'Evaluate': 0.024774953722953796, 'Create': 0.01853260211646557}}


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
